In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [45]:
df = pd.read_csv(r"Final_Processed_Dataset.csv")
df.head()

,News_ID,Title,News_Body,Stance,Label,issue,topic,roundup,image
0,0,North Carolina House race carries unlikely han...,"CHARLOTTE, N.C. — The closely-watched special ...",lean left,Liberal,Illegal Ballot Harvesting Prompted Re-Do of No...,Voting Rights and Voter Fraud,"CHARLOTTE, N.C. — The closely-watched special ...",0_left_2.jpg
1,0,North Carolina House race carries unlikely han...,"CHARLOTTE, N.C. — The closely-watched special ...",lean left,Liberal,Illegal Ballot Harvesting Prompted Re-Do of No...,Voting Rights and Voter Fraud,"CHARLOTTE, N.C. — The closely-watched special ...",0_left_1.jpg
2,0,Trump unloads on 'disloyal' Democratic House c...,High stakes were matched by some of President ...,lean right,Conservative,Illegal Ballot Harvesting Prompted Re-Do of No...,Voting Rights and Voter Fraud,High stakes were matched by some of President ...,0_right_5.jpg
3,0,Trump unloads on 'disloyal' Democratic House c...,High stakes were matched by some of President ...,lean right,Conservative,Illegal Ballot Harvesting Prompted Re-Do of No...,Voting Rights and Voter Fraud,High stakes were matched by some of President ...,0_right_4.jpg
4,0,Trump unloads on 'disloyal' Democratic House c...,High stakes were matched by some of President ...,lean right,Conservative,Illegal Ballot Harvesting Prompted Re-Do of No...,Voting Rights and Voter Fraud,High stakes were matched by some of President ...,0_right_1.jpg


In [46]:
list(df)

['News_ID',
 'Title',
 'News_Body',
 'Stance',
 'Label',
 'issue',
 'topic',
 'roundup',
 'image']

In [47]:
df = df.drop(columns=["image"])
# Dropping Images Column for Uni-modal Detection

## HTML Tag Cleaning

In [48]:
import re
def striphtml(data):
  p = re.compile(r'<.*?>')
  return p.sub('',data)

In [49]:
cols_to_clean = [
 'Title',
 'News_Body',
 'roundup']

# Apply the function to all selected columns, handling NaN values
for col_name in cols_to_clean:
    df[col_name] = df[col_name].fillna('').apply(striphtml)

## LowerCasing





In [50]:
# Define the target columns
cols_to_lower = ['Title', 'News_Body',
    'Stance', 'Label', 'issue',
    'topic', 'roundup'
]

# Apply lowercasing using the .str accessor
for col in cols_to_lower:
    # We cast to string first to handle any mixed types (like News_ID)
    # then apply lower()
    df[col] = df[col].astype(str).str.lower()

# Quick check to verify
print(df[cols_to_lower].head())

                                               Title  \
0  north carolina house race carries unlikely han...   
1  north carolina house race carries unlikely han...   
2  trump unloads on 'disloyal' democratic house c...   
3  trump unloads on 'disloyal' democratic house c...   
4  trump unloads on 'disloyal' democratic house c...   

                                           News_Body      Stance  \
0  charlotte, n.c. — the closely-watched special ...   lean left   
1  charlotte, n.c. — the closely-watched special ...   lean left   
2  high stakes were matched by some of president ...  lean right   
3  high stakes were matched by some of president ...  lean right   
4  high stakes were matched by some of president ...  lean right   

          Label                                              issue  \
0       liberal  illegal ballot harvesting prompted re-do of no...   
1       liberal  illegal ballot harvesting prompted re-do of no...   
2  conservative  illegal ballot harvesting p

## Noise Cleaning

In [51]:
import re

def clean_for_transformer(text):
    if not isinstance(text, str):
        return ""

    # 1. Remove digital noise but keep sentence markers
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # Remove URLs
    text = re.sub(r'@\S+', '', text)                  # Remove Mentions

    # 2. Keep only alphanumeric and basic punctuation [.,!?;:]
    # This allows the model to see the "structure" of the news article
    text = re.sub(r'[^a-zA-Z0-9\s.,!?;:]', '', text)

    # 3. Standardize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [52]:
df.sample()

,News_ID,Title,News_Body,Stance,Label,issue,topic,roundup
1737,438,the state we're in: will the pandemic revoluti...,the state has been in retreat since the 80s he...,lean left,liberal,the role of government during the coronavirus ...,role of government,the state has been in retreat since the 80s he...


## Remove Duplicate Records

In [53]:
# Check how many duplicates exist before removing
duplicate_count = df.duplicated().sum()
print(f"Total exact row duplicates found: {duplicate_count}")

Total exact row duplicates found: 826


In [54]:
# Remove exact duplicates (keeps the first occurrence)
df = df.drop_duplicates(keep='first')

# Optional: Remove duplicates based only on the News_Body
# (Highly recommended if titles vary slightly but content is identical)
df = df.drop_duplicates(subset=['News_Body'], keep='first')

print(f"Rows remaining after removal: {len(df)}")

Rows remaining after removal: 9190


In [55]:
df = df.reset_index(drop=True)

In [56]:
df.head(10)

,News_ID,Title,News_Body,Stance,Label,issue,topic,roundup
0,0,north carolina house race carries unlikely han...,"charlotte, n.c. — the closely-watched special ...",lean left,liberal,illegal ballot harvesting prompted re-do of no...,voting rights and voter fraud,"charlotte, n.c. — the closely-watched special ..."
1,0,trump unloads on 'disloyal' democratic house c...,high stakes were matched by some of president ...,lean right,conservative,illegal ballot harvesting prompted re-do of no...,voting rights and voter fraud,high stakes were matched by some of president ...
2,0,illegal ballot harvesting caused a do-over hou...,a u.s. house special election in north carolin...,center,center,illegal ballot harvesting prompted re-do of no...,voting rights and voter fraud,a u.s. house special election in north carolin...
3,1,republicans hold on to a north carolina congre...,dan bishop’s narrow win suggests that republic...,left,liberal,republican dan bishop wins special election in...,elections,dan bishop’s narrow win suggests that republic...
4,1,'voters said no to radical liberal policies': ...,republicans scored an important victory in nor...,right,conservative,republican dan bishop wins special election in...,elections,republicans scored an important victory in nor...
5,1,trump-backed republican wins north carolina sp...,republican dan bishop won a special u.s. congr...,center,center,republican dan bishop wins special election in...,elections,republican dan bishop won a special u.s. congr...
6,2,trump administration considering ban on flavor...,president trump on wednesday gathered reporter...,left,liberal,president trump considering regulation on vape...,white house,president trump on wednesday gathered reporter...
7,2,"trump takes aim at teen vaping, proposes ban o...","in an effort to curb teen vaping, president tr...",lean right,conservative,president trump considering regulation on vape...,white house,"in an effort to curb teen vaping, president tr..."
8,2,trump seeks ban on flavored e-cigarettes,the trump administration is seeking to ban all...,center,center,president trump considering regulation on vape...,white house,the trump administration is seeking to ban all...
9,3,a letter to 9/11: happy birthday to the war wi...,the day you entered this world i woke to the n...,lean left,liberal,america remembers 9/11 after 18 years,terrorism,the day you entered this world i woke to the n...


## Spell Checking

In [57]:
!pip install tqdm
from tqdm import tqdm
tqdm.pandas()


In [58]:
# for col in cols_to_spellcheck:
#     df[col] = df[col].progress_apply(correct_spelling)

In [59]:
# from textblob import TextBlob

# # Columns you've identified for the project
# cols_to_spellcheck = [
#     'Title', 'News_Body', 'Stance',
#     'Label', 'issue', 'topic', 'roundup'
# ]

# def correct_spelling(text):
#     # Check if the text is a valid string and not empty
#     if isinstance(text, str) and text.strip():
#         # TextBlob.correct() returns a 'Blob' object, so we cast to string
#         return str(TextBlob(text).correct())
#     return text

# # Apply the correction
# for col in cols_to_spellcheck:
#     print(f"Correcting spelling in {col}... This may take a while.")
#     df[col] = df[col].apply(correct_spelling)

# # Verify the result
# print(df[cols_to_spellcheck].head())

In [60]:
df.sample()

,News_ID,Title,News_Body,Stance,Label,issue,topic,roundup
5120,1707,trump condemns shooting as a ‘wicked act’ of a...,president trump denounced the mass shooting in...,center,center,11 people killed in pittsburgh synagogue,white house,president trump denounced the mass shooting in...


In [61]:
pip install -U symspellpy

In [62]:
import pkg_resources
from symspellpy import SymSpell, Verbosity

# 1. Initialize SymSpell with max edit distance of 2
sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)

# 2. Load the included English frequency dictionary
dictionary_path = pkg_resources.resource_filename(
    "symspellpy", "frequency_dictionary_en_82_765.txt"
)
# term_index=0 (word), count_index=1 (frequency)
sym_spell.load_dictionary(dictionary_path, term_index=0, count_index=1)

# 3. Define the fast correction function
def symspell_correct(text):
    if not isinstance(text, str) or not text.strip():
        return text

    # lookup_compound handles multi-word sentences and preserves spaces
    suggestions = sym_spell.lookup_compound(text, max_edit_distance=2)
    return suggestions[0].term

# 4. Apply to your bias project columns
cols_to_fix = ['Title', 'News_Body', 'Stance', 'Label', 'issue', 'topic', 'roundup']
for col in cols_to_fix:
    print(f"Fast-correcting {col}...")
    df[col] = df[col].apply(symspell_correct)

Fast-correcting Title...
Fast-correcting News_Body...
Fast-correcting Stance...
Fast-correcting Label...
Fast-correcting issue...
Fast-correcting topic...
Fast-correcting roundup...


In [63]:
df.sample()

,News_ID,Title,News_Body,Stance,Label,issue,topic,roundup
5654,1886,us supreme court allows trump military transge...,the united states supreme court has allowed pr...,center,center,supreme court allows partial ban on transgende...,supreme court,the united states supreme court has allowed pr...


## Downloading the Dataset and Converting into CSV

In [64]:
from google.colab import files

# 1. Save the dataframe to the current directory in Colab
df.to_csv('cleaned_political_bias_data.csv', index=False)

# 2. Download the file to your computer
files.download('cleaned_political_bias_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [70]:
# Access row index 1, column name "News_Body"
print(df.loc[5100, "roundup"])

halloween is always getting people in trouble there's just something about feeling liberated to dress in costume that makes people stupid i hate to bring this up but remember how now beloved
